In [ ]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from robusta_hmf import Robusta
import time
#%matplotlib widget

In [ ]:
#define constants
c = 299792.458 # km/s
LN10 = np.log(10.)
MAX_IVAR = 2.5e3
MIN_IVAR = 0.1
H_ALPHA, H_BETA = 6564.614, 4862.721 #Reference: from classic.sdss.org
HE_I = 6680 #Reference: Johanna brain
HE_II = 4686 #Reference: Johanna brain

lines = [H_ALPHA, H_BETA, HE_I, HE_II]
line_labels = ["Halpha", "Hbeta", "HeI", "HeII"]
moment_names = ["EW", "centroid", "variance", "skew", "m4"]

H_delta_lnlam_line = 1000 / c 
H_delta_loglam_line = H_delta_lnlam_line / LN10 
HE_delta_lnlam_line = 500 / c
HE_delta_loglam_line = HE_delta_lnlam_line / LN10

half_wids = [H_delta_loglam_line, H_delta_loglam_line, HE_delta_loglam_line, HE_delta_loglam_line] 

In [ ]:
#read in the data and deal with radial velocities
with open('spectra_5021_2026-07-17_data.pkl','rb') as f:
     pkl_data = pickle.load(f)

In [ ]:
fluxes, loglam, ivars, continuua, spec_files, rvs, ews_halphas, nana_hash, parent_spectra = pkl_data
print(fluxes.shape, loglam.shape, ivars.shape, continuua.shape, len(spec_files), rvs.shape, ews_halphas.shape, nana_hash.shape, parent_spectra.shape)

In [ ]:
def apply_ew_mask(pickle_data, ew_limit = 0):
    
    fluxes, loglam, ivars, continuua, spec_files, rvs, ews_halphas, nana_hash, parent_spectra = pkl_data
    good_ew_mask = ews_halphas >= ew_limit

    print(f"Keeping {good_ew_mask.sum()} / {len(good_ew_mask)} spectra (removed {(~good_ew_mask).sum()} with negative EW)")

    print(nana_hash.shape)
    
    return (fluxes[good_ew_mask], ivars[good_ew_mask], continuua[good_ew_mask], rvs[good_ew_mask], ews_halphas[good_ew_mask], 
           nana_hash[good_ew_mask],[sf for sf, keep in zip(spec_files, good_ew_mask) if keep], parent_spectra[good_ew_mask])

In [ ]:
fluxes, ivars, continuua, rvs, ews_halphas, nana_hash, spec_files, parent_spectra = apply_ew_mask(pkl_data)

In [ ]:
#idk what's going on with the error here
print(fluxes.shape, loglam.shape, ivars.shape, continuua.shape, len(spec_files), rvs.shape, ews_halphas.shape, nana_hash.shape, parent_spectra.shape)

In [ ]:
#log lambda values have been corrupted

def get_delta_log_lam(loglambdas):
    return np.median(loglambdas[1:] - loglambdas[:-1])

def get_lambdas(loglambdas):
    return 10**loglambdas

#trial
delta_log_lambda_pixel = get_delta_log_lam(loglam)
wave = get_lambdas(loglam)

In [ ]:
def pixel_shift(flxs, ivrs, contins, rad_vels, log_lambda):

    delta_log_lambda_pixel = get_delta_log_lam(log_lambda)
    
    delta_log_lambdas = (rad_vels / c) / LN10
    delta_pixels = np.round(delta_log_lambdas/delta_log_lambda_pixel).astype(int)

    rest_flxs = np.zeros_like(flxs) + 1.
    rest_ivrs = np.zeros_like(ivrs)
    rest_contins = np.zeros_like(contins) + np.nan

    for i, dp in enumerate(delta_pixels):
        if dp < 0:
            rest_flxs[i, -dp:] = flxs[i, :dp]
            rest_ivrs[i, -dp:] = ivrs[i, :dp]
            rest_contins[i, -dp:] = contins[i, :dp]
            
        elif dp > 0:
            rest_flxs[i, :-dp] = flxs[i, dp:]
            rest_ivrs[i, :-dp] = ivrs[i, dp:]
            rest_contins[i, :-dp] = contins[i, dp:]
            
        else:
            rest_flxs[i, :] = flxs[i, :]
            rest_ivrs[i, :] = ivrs[i, :]
            rest_contins[i, :] = contins[i, :]

    return rest_flxs, rest_ivrs, rest_contins

In [ ]:
rest_fluxes, rest_ivars, rest_continuua = pixel_shift(fluxes, ivars, continuua, rvs, loglam)

In [ ]:
def fix_nans_and_infinites(rest_flxs, rest_ivrs):

    #fix nans and infinities
    bad = np.logical_not(np.isfinite(rest_flxs))
    rest_flxs[bad] = 1
    rest_ivrs[bad] = 0
    
    bad = np.logical_not(np.isfinite(rest_ivrs))
    rest_flxs[bad] = 1
    rest_ivrs[bad] = 0
    
    bad = np.logical_or((rest_flxs > 2.0), (rest_flxs < 0))
    rest_flxs[bad] = 1
    rest_ivrs[bad] = 0
    
    bad = rest_ivrs > MAX_IVAR
    # rest_fluxes[bad] = 1
    # rest_ivars[bad] = 0
    rest_ivrs[bad] = MAX_IVAR
    
    bad = rest_ivrs < MIN_IVAR
    rest_flxs[bad] = 1.0
    rest_ivrs[bad] = MIN_IVAR

    return rest_flxs, rest_ivrs

In [ ]:
data, weights = fix_nans_and_infinites(rest_fluxes, rest_ivars)

In [ ]:
# split data to A and B using nana_hash parity, only train on A
def split_and_train(rest_flxs, rest_ivrs, n_hash, K = 12, scale = 1, nu = 1):
    
    N, M = rest_flxs.shape
    
    Ainx = np.where(n_hash % 2 == 0)[0]
    Binx = np.where(n_hash % 2 == 1)[0]

    print(f"A: {len(Ainx)} spectra, B: {len(Binx)} spectra")

    modA = Robusta(rank=K, robust=True, robust_scale = scale, robust_nu = nu)
    start = time.perf_counter()
    modA.fit(rest_flxs[Ainx], rest_ivrs[Ainx], max_iter=10000)
    end = time.perf_counter()
    print("total time:", (end - start)/60, "minutes")

    return modA, Ainx, Binx    

In [ ]:
modelA, Aindx, Bindx = split_and_train(data, weights, nana_hash)

In [ ]:
def censor_and_synthesize(lines, log_lambdas, rest_fluxes, rest_ivars, Binx, modA):

    log_lines = np.log10(lines)
    
    log_H_ALPHA, log_H_BETA = log_lines[0], log_lines[1]
    log_HE_I, log_HE_II = log_lines[2], log_lines[3]
    
    region_alpha = np.abs(log_lambdas - log_H_ALPHA)
    region_beta = np.abs(log_lambdas - log_H_BETA)
    
    region_he_i = np.abs(log_lambdas - log_HE_I)
    region_he_ii = np.abs(log_lambdas - log_HE_II)
    
    censor_mask = np.ones_like(log_lambdas)
    censor_mask[(region_alpha < H_delta_loglam_line) | (region_beta < H_delta_loglam_line) 
              | (region_he_i < HE_delta_loglam_line) | (region_he_ii < HE_delta_loglam_line)] = 0 #?
    
    state, _ = modA.infer(rest_fluxes[Binx], rest_ivars[Binx] * censor_mask[None, :])
    synth_B = modA.synthesize(state)

    return state, synth_B

In [ ]:
state, synth_B = censor_and_synthesize(lines, loglam, data, weights, Bindx, modelA)

In [ ]:
def get_moment_vals(log_lambda, rest_fluxes, rest_ivars, Binx, synthB):
    
    delta_log_lambda_pixel = get_delta_log_lam(log_lambda)


    lam = 10 ** log_lambda

    exponents = np.arange(5)
    moments = np.zeros((N, len(lines), len(exponents))) + np.nan
    moment_errs = np.zeros((N, len(lines), len(exponents))) + np.nan

    for j, (line, wid) in enumerate(zip(lines, half_wids)):
        logline = np.log10(line)
        integration_weight = delta_log_lambda_pixel * LN10 * lam * (np.abs(logline - log_lambda) < wid)
        resid = rest_fluxes[Binx] - synthB
        diff = lam - line #this is hogg's lam-lam0 he wrote about
    
        #compute vals and corresponding errors
        for k, expo in enumerate(exponents):
            something = integration_weight * diff ** expo
            moments[:, j, k] = np.sum(resid * something[None, :], axis=1)
            moment_errs[:, j, k] = np.sqrt(np.sum(something[None, :] ** 2 / rest_ivars[Binx], axis=1))

    return moments, moment_errs


In [ ]:
moments, moment_errs = get_moment_vals(loglam, data, weights, Bindx, synth_B)

print(moments, moment_errs)

In [ ]:
### functions for graphing

def plot_eigenspectra(modA, log_lambda, K = 12):
    eigsA = modA.basis_vectors().T
    wave = 10 ** log_lambda

    f = plt.figure(figsize=(15, 15))
    plt.title("Eigenspectra")
    offset = 0.2
    for i in range(K):
        f.gca().plot(wave, eigsA[i] + i * offset)
    
    #plt.axvline(H_ALPHA, lw=1, alpha=0.5, color="blue")
    plt.xlabel("Wavelength (angstrom)")
    plt.show()
    plt.close()


def plot_synthesized(synthB, rest_fluxes, Binx, log_lambda, lines, K = 12):
    
    f = plt.figure(figsize=(15, 15))
    wave = 10 ** log_lambda
    
    offset = 0.5
    for i in range(12):
        f.gca().plot(wave, rest_fluxes[Binx[i]] + i * offset, color="k")
        f.gca().plot(wave, synthB[i] + i * offset, color="r")

    for line in lines: 
        plt.axvline(line, lw=1, alpha=0.5, color="b")
    plt.xlim(4200, 7000)
    plt.xlabel("Wavelength (angstrom)")
    plt.show()
    plt.close()


#make def for the residual plotting

In [ ]:
plot_eigenspectra(modelA, loglam)
plot_synthesized(synth_B, data, Bindx, loglam, lines)